In [1]:
import cv2
import numpy as np
import os
import glob
import shutil

def generate_contrastive_dataset(input_dirs, output_base_dir):
    """
    Takes a list of input directories of Red Trucks and generates paired datasets 
    in Red (Original), White, Yellow, and Blue.
    """
    # Define target colors, including the original 'Red'
    colors = ['Red', 'White', 'Yellow', 'Blue']

    # Create output directories
    for color in colors:
        os.makedirs(os.path.join(output_base_dir, color), exist_ok=True)

    # Ensure input_dirs is a list to handle glob results easily
    if isinstance(input_dirs, str):
        input_dirs = [input_dirs]

    # Loop through all provided input directories
    for input_dir in input_dirs:
        print(f"Processing directory: {input_dir}")
        
        if not os.path.exists(input_dir):
            print(f"Warning: Directory {input_dir} does not exist. Skipping.")
            continue

        # Loop through all images in the input directory
        for filename in os.listdir(input_dir):
            if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue

            filepath = os.path.join(input_dir, filename)
            
            # 1. Save the Original Image into the 'Red' folder
            shutil.copy(filepath, os.path.join(output_base_dir, 'Red', filename))

            # Read the image for processing
            img = cv2.imread(filepath)
            if img is None:
                print(f"Warning: Could not read {filename} with OpenCV")
                continue

            # Convert to HSV color space for accurate color manipulation
            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

            # Isolate the Red Color
            lower_red1 = np.array([0, 70, 50])
            upper_red1 = np.array([10, 255, 255])
            mask1 = cv2.inRange(hsv, lower_red1, upper_red1)

            lower_red2 = np.array([170, 70, 50])
            upper_red2 = np.array([180, 255, 255])
            mask2 = cv2.inRange(hsv, lower_red2, upper_red2)

            # Combine masks to get all red pixels
            red_mask = mask1 | mask2

            # 2. Generate White Truck
            hsv_white = hsv.copy()
            hsv_white[:, :, 1] = np.where(red_mask > 0, 0, hsv_white[:, :, 1]) 
            v_channel = hsv_white[:, :, 2].astype(np.float32)
            v_channel = np.where(red_mask > 0, np.clip(v_channel * 1.2 + 20, 0, 255), v_channel)
            hsv_white[:, :, 2] = v_channel.astype(np.uint8)
            
            white_img = cv2.cvtColor(hsv_white, cv2.COLOR_HSV2BGR)
            cv2.imwrite(os.path.join(output_base_dir, 'White', filename), white_img)

            # 3. Generate Yellow Truck
            hsv_yellow = hsv.copy()
            hsv_yellow[:, :, 0] = np.where(red_mask > 0, 30, hsv_yellow[:, :, 0])
            yellow_img = cv2.cvtColor(hsv_yellow, cv2.COLOR_HSV2BGR)
            cv2.imwrite(os.path.join(output_base_dir, 'Yellow', filename), yellow_img)

            # 4. Generate Blue Truck
            hsv_blue = hsv.copy()
            hsv_blue[:, :, 0] = np.where(red_mask > 0, 120, hsv_blue[:, :, 0])
            blue_img = cv2.cvtColor(hsv_blue, cv2.COLOR_HSV2BGR)
            cv2.imwrite(os.path.join(output_base_dir, 'Blue', filename), blue_img)

    print(f"✅ Processing complete! Contrastive dataset generated in: {output_base_dir}")

# --- Execution ---
if __name__ == "__main__":
    # Your Kaggle dataset path
    search_path = "/kaggle/input/datasets/vitaliykinakh/stable-imagenet1k/imagenet1k/555_*"
    dirs = sorted(glob.glob(search_path))

    OUTPUT_FOLDER = "./paired_truck_dataset"
    ZIP_FILE_NAME = "./paired_truck_dataset_ready" # Shutil will add the .zip extension automatically
    
    if not dirs:
        print(f"Error: No directories found matching '{search_path}'")
    else:
        # 1. Generate the modified images and copy originals
        generate_contrastive_dataset(dirs, OUTPUT_FOLDER)
        
        # 2. Zip the entire output folder for easy download
        print(f"📦 Zipping dataset to {ZIP_FILE_NAME}.zip...")
        shutil.make_archive(ZIP_FILE_NAME, 'zip', OUTPUT_FOLDER)
        print("✅ Zipping complete! You can now download the file from your working directory.")

Processing directory: /kaggle/input/datasets/vitaliykinakh/stable-imagenet1k/imagenet1k/555_fire engine, fire truck
✅ Processing complete! Contrastive dataset generated in: ./paired_truck_dataset
📦 Zipping dataset to ./paired_truck_dataset_ready.zip...
✅ Zipping complete! You can now download the file from your working directory.


In [2]:
from IPython.display import FileLink; FileLink(r'paired_truck_dataset_ready.zip')

/kaggle/working/paired_truck_dataset_ready.zip